# 03 — Public Data, Provenance, and Leakage-Aware Partitions

## Objective
Download or inspect original PubMedQA records, explain the nested abstract-context schema, verify source labels and publication identifiers, and produce a stable train/validation/test manifest. The manifest is reused by all model-training and comparison notebooks.

## Research protocol
- PubMedQA `pqa_labeled` consists of labeled research questions with evidence abstracts and one of three decisions: **yes**, **no**, or **maybe**. The source provides real, human-labeled records; its provenance must be retained.
- The **publication identifier** (`pubid`) is used as a grouping key. Every occurrence of a publication belongs to one partition.
- SHA-256 (Secure Hash Algorithm 256-bit) deterministically maps publication IDs into approximate 80%/10%/10% partitions. These are project-specific partitions, **not** the official published PubMedQA benchmark split.
- Once written, the manifest is checked rather than silently replaced. Data revisions or changed split rules require an explicitly new experiment protocol.

Downloading data and training a model require network access. The examples below remain executable without downloads by default.

### Research workspace
The output path is printed before writing files so run artifacts are traceable.

In [1]:
# Locate the research workspace from the repository root or the notebooks folder.
from pathlib import Path
cwd = Path.cwd().resolve()
if (cwd / "exploration" / "notebooks").is_dir():
    RESEARCH = cwd / "exploration"
elif cwd.name == "notebooks" and cwd.parent.name == "exploration":
    RESEARCH = cwd.parent
elif cwd.name == "exploration" and (cwd / "notebooks").is_dir():
    RESEARCH = cwd
else:
    raise FileNotFoundError("Open Jupyter from the repository root, exploration/, or exploration/notebooks/.")

RUNS = RESEARCH / "runs"
RUNS.mkdir(parents=True, exist_ok=True)
print("Research workspace:", RESEARCH)
print("Experiment outputs:", RUNS)

Research workspace: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration
Experiment outputs: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration\runs


### Dataset schema and validation functions
Every relevant field and split assignment is visible in the following cell. The SHA-256 assignment does not use answer labels or prompt text.

In [2]:
# PubMedQA dataset protocol: the same source, group assignment and prompt data
# must be used in notebooks 03, 04, 05, 06, 11, 12 and 18.
import hashlib
import json
import os
from collections import Counter


DATASET_NAME = "qiaojin/PubMedQA"
DATASET_CONFIG = "pqa_labeled"      # Explicitly labeled PubMedQA questions.
SEED = 42
LABELS = ("yes", "no", "maybe")
MANIFEST = RESEARCH / "data" / "pubmedqa_split_manifest.json"
MANIFEST.parent.mkdir(parents=True, exist_ok=True)


def paper_id(row):
    """Stable publication ID used to prevent paper-level train/evaluation overlap."""
    value = str(row.get("pubid", "")).strip()
    if not value:
        raise ValueError("Missing PubMedQA publication ID; cannot split safely.")
    return value


def evidence_text(row):
    """Extract abstract sentences from PubMedQA's nested context object."""
    context = row.get("context") or {}
    sentences = context.get("contexts", []) if isinstance(context, dict) else []
    if not isinstance(sentences, list):
        raise TypeError("Expected context['contexts'] to be a list of abstract sentences.")
    return "\n".join(sentence.strip() for sentence in sentences if isinstance(sentence, str))


def assign_split(identifier):
    """Deterministically assign ALL rows with one publication ID to the same split."""
    digest = hashlib.sha256(f"{SEED}:{identifier}".encode("utf-8")).digest()
    fraction = int.from_bytes(digest[:8], "big") / 2**64
    if fraction < 0.80:
        return "train"
    if fraction < 0.90:
        return "validation"
    return "test"


def source_fingerprint(rows):
    """Hash canonical source fields to detect modifications between experiments."""
    canonical = sorted(
        (paper_id(row), str(row["question"]), evidence_text(row), str(row["final_decision"]).lower())
        for row in rows
    )
    payload = json.dumps(canonical, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def prepare_partitions(rows, create_manifest=False):
    """Check labels, record stable splits and require identical upstream source data."""
    records = list(rows)
    identifiers = [paper_id(row) for row in records]
    if len(identifiers) != len(set(identifiers)):
        raise ValueError("Repeated publication IDs: review source records before evaluation.")
    if any(str(row["final_decision"]).lower() not in LABELS for row in records):
        raise ValueError("Unexpected label; expected one of yes/no/maybe.")
    if any(not evidence_text(row) for row in records):
        raise ValueError("Empty evidence context; review source-data schema.")

    partitions = {name: [] for name in ("train", "validation", "test")}
    for row in records:
        partitions[assign_split(paper_id(row))].append(row)
    metadata = {
        "dataset": DATASET_NAME,
        "config": DATASET_CONFIG,
        "split_algorithm": "SHA256(seed:pubid), buckets 0-0.8/0.8-0.9/0.9-1.0",
        "seed": SEED,
        "fingerprint": source_fingerprint(records),
        "ids": {name: sorted(paper_id(row) for row in subset) for name, subset in partitions.items()},
    }
    if MANIFEST.exists():
        previous = json.loads(MANIFEST.read_text(encoding="utf-8"))
        if previous != metadata:
            raise RuntimeError("Dataset differs from saved split manifest. Do not silently overwrite it; inspect data revision and protocol.")
    elif create_manifest:
        MANIFEST.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
        print("Created reproducible split manifest:", MANIFEST)
    else:
        raise FileNotFoundError("Split manifest missing. Run notebook 03 with RUN_PUBLIC_DATA=1 first.")

    for name, subset in partitions.items():
        print(f"{name}: n={len(subset)}, labels={dict(Counter(str(r['final_decision']).lower() for r in subset))}")
    if min(len(p) for p in partitions.values()) == 0:
        raise ValueError("At least one split is empty; check dataset size and split rules.")
    return partitions

### Worked split calculation
A deterministic example verifies that the same publication never changes partition. This is a unit check, not a biomedical dataset result.

In [3]:
example_ids = ["PMID-100", "PMID-100", "PMID-200", "PMID-300"]
example_splits = [(identifier, assign_split(identifier)) for identifier in example_ids]
print("Illustrative publication-to-split mapping:", example_splits)
assert example_splits[0][1] == example_splits[1][1]

Illustrative publication-to-split mapping: [('PMID-100', 'train'), ('PMID-100', 'train'), ('PMID-200', 'train'), ('PMID-300', 'train')]


### Original source retrieval and manifest creation
Set `RUN_PUBLIC_DATA=1` before executing this cell to inspect the public source. With `HF_DATASETS_OFFLINE=1`, a single existing Hugging Face Arrow cache is read directly; otherwise the source is loaded through Hugging Face Datasets. A subsequent run verifies its fingerprint and partition identifiers against the saved manifest. The cell prints the dataset schema, class distributions, and ID counts.

In [4]:
if os.getenv("RUN_PUBLIC_DATA", "0") == "1":
    from datasets import Dataset, config, load_dataset

    if os.getenv("HF_DATASETS_OFFLINE", "0") == "1":
        cache_root = Path(config.HF_DATASETS_CACHE)
        cached_files = list(cache_root.glob("qiaojin___pub_med_qa/pqa_labeled/*/*/pub_med_qa-train.arrow"))
        if len(cached_files) != 1:
            raise FileNotFoundError("Expected one cached PubMedQA labeled Arrow file; found " + str(len(cached_files)))
        source = Dataset.from_file(str(cached_files[0]))
        print("Source mode: existing Hugging Face Arrow cache")
        print("Cache file:", cached_files[0])
    else:
        source = load_dataset(DATASET_NAME, DATASET_CONFIG, split="train")
        print("Source mode: Hugging Face load_dataset")
    print("Dataset columns:", source.column_names)
    print("Original record count:", len(source))
    print("Example identifier:", paper_id(source[0]))
    print("Example abstract sentence count:", len(source[0]["context"]["contexts"]))

    partitions = prepare_partitions(source, create_manifest=True)
    print("Saved source fingerprint:", source_fingerprint(source))
    print("Manifest:", MANIFEST)
else:
    print("No network access requested. Set RUN_PUBLIC_DATA=1 to inspect and partition PubMedQA.")

Source mode: existing Hugging Face Arrow cache
Cache file: C:\Users\weiju\.cache\huggingface\datasets\qiaojin___pub_med_qa\pqa_labeled\0.0.0\9001f2853fb87cab8d220904e0de81ac6973b318\pub_med_qa-train.arrow
Dataset columns: ['pubid', 'question', 'context', 'long_answer', 'final_decision']
Original record count: 1000
Example identifier: 21645374
Example abstract sentence count: 2
train: n=775, labels={'yes': 433, 'no': 255, 'maybe': 87}
validation: n=121, labels={'no': 40, 'yes': 64, 'maybe': 17}
test: n=104, labels={'yes': 55, 'no': 43, 'maybe': 6}
Saved source fingerprint: 26f50b35963ad97e7e379add5f7ad1fdb5c82f87e831fa58ee46e4f44f0e9858
Manifest: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration\data\pubmedqa_split_manifest.json
